# NB-B1 - FF++ visual pretraining (CLIP ViT-L/14, LayerNorm-tuned)

Occupies the slot the **cut SBI stage** was built for, and writes the same
`{'backbone_state', 'meta'}` checkpoint format, so `load_visual_encoder` in
`crossfuse_v5.py` consumes it unchanged.

### Why this recipe

Two independent papers converge on it, both training on FF++ c23:

| | backbone | tuned params | Celeb-DF v2 | DFDC |
|---|---|---|---|---|
| LNCLIP-DF ([arXiv:2508.06248](https://arxiv.org/abs/2508.06248)) | CLIP ViT-L/14 | LayerNorm only (0.03%) | **96.5** | **87.0** |
| Effort (ICML 2025, [arXiv:2411.15633](https://arxiv.org/abs/2411.15633)) | CLIP ViT-L/14 | SVD-orthogonal residual | 95.6 | 85.4 |
| **CrossFuse v5 (paper)** | EfficientNet-B4 / ImageNet | full fine-tune | 64.0 | 58.0 |

LNCLIP-DF's own ablation on Celeb-DF v2: linear probe 78.1 -> **+LN-tuning
94.9** -> +L2-norm 96.2. Full fine-tuning and LoRA both *overfit* - LoRA hit
99.99% train AUROC in 2 epochs. So the gain is specifically from tuning
*very little*, which is also why this is cheap.

### Preregistered gate

`FFPP_CONFIG["GATE_CELEBDF_AUC"] = 0.85` and `GATE_DFDC_AUC = 0.75`, fixed
**before** the run in the same spirit as v5's `SYNC_MIN_AUC = 0.70`. If the
encoder misses them, it does **not** proceed to NB-B - the reproduction has
failed and that is the finding to report.

**Attach:** `crossfuse-v5-lib`, `ffpp-crops-v5` (from NB-A3), Celeb-DF v2, DFDC
sample. **Internet must be ON** (CLIP weights download on first use).

### Result (2026-09)

**Ran and passed the gate:** zero-shot Celeb-DF **0.9186** (gate 0.85, baseline 0.6399) and DFDC **0.8494** (gate 0.75, baseline 0.5799). Published as Kaggle Dataset `ffpp-encoder-v5` (`ffpp_encoder_v5.pth` + `ffpp_zeroshot_v5.json`), consumed by `B-train`. `open_clip_torch` is installed `--no-deps` (see cell 1) so pip cannot disturb Kaggle's torch/torchvision pairing.

In [ ]:
# open_clip_torch's real deps (checked against its PyPI wheel metadata):
# torch, torchvision, regex, ftfy, tqdm, huggingface-hub, safetensors,
# timm>=1.0.17. torch/torchvision/tqdm/huggingface-hub are already on Kaggle's
# base image; installing the rest with --no-deps (timm and safetensors
# included -- neither pins a torch/torchvision version, checked directly
# against their wheel metadata too) is what actually keeps pip from ever
# touching the preinstalled torch/torchvision pairing, instead of hoping a
# hand-picked subset happens to cover what open_clip.__init__ imports.
!pip install -q --no-deps open_clip_torch timm safetensors ftfy regex pyyaml
!pip install -q --no-deps facenet-pytorch
import sys, glob, os
_lib_hits = glob.glob("/kaggle/input/**/crossfuse_v5.py", recursive=True)
assert _lib_hits, ("crossfuse_v5.py not found under /kaggle/input -- attach the "
                   "crossfuse-v5-lib dataset in Add Input, then restart the session.")
sys.path.insert(0, os.path.dirname(_lib_hits[0]))
print("Using crossfuse_v5 library from:", _lib_hits[0])

# Verify the two imports that actually matter, HERE, before anything downstream
# depends on them silently being None -- catches a broken install in seconds
# instead of ~30min into train_ffpp_encoder.
import torch, torchvision
print(f"torch {torch.__version__} | torchvision {torchvision.__version__}")
from torchvision.models import efficientnet_b0  # noqa: F401 -- import-only check
print("torchvision.models.efficientnet_b0 -- OK")
import open_clip  # noqa: F401 -- import-only check
print("open_clip -- OK")

In [ ]:
import csv, json
import numpy as np
import torch
import matplotlib.pyplot as plt

from crossfuse_v5 import CONFIG, seed_everything, safe_auc, make_rng
import ffpp_v5 as FF

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(CONFIG["SEED"])
print(f"device: {device} | GPUs: {torch.cuda.device_count()}")

_m = glob.glob("/kaggle/input/**/ffpp_crops_v5/manifest.csv", recursive=True) or \
     glob.glob("/kaggle/input/**/manifest.csv", recursive=True)
CROP_DIR = None
for p in _m:
    with open(p) as f:
        head = f.readline()
    if "method" in head and "target_id" in head:   # the FF++ manifest, not FakeAVCeleb's
        CROP_DIR = os.path.dirname(p)
        break
assert CROP_DIR, ("FF++ manifest.csv not found -- attach the ffpp-crops-v5 dataset "
                  "produced by NB-A3.")
print("FF++ crops:", CROP_DIR)

with open(os.path.join(CROP_DIR, "manifest.csv")) as f:
    ROWS = list(csv.DictReader(f))
for r in ROWS:
    r["label"] = int(r["label"])
n_fake = sum(r["label"] for r in ROWS)
print(f"{len(ROWS)} rows | real {len(ROWS) - n_fake} | fake {n_fake}")

In [ ]:
# ---- Split. Prefer FF++'s OFFICIAL json; the fallback is measured, not assumed ----
# FF++ pairs its 1000 videos by a permutation (each id is a target once and a
# source once), so union-find over {target, source} follows that permutation's
# CYCLES -- and a random permutation's largest cycle typically covers ~60% of
# its elements. Strict transitive closure therefore collapses most of FF++ into
# one component and cannot produce a split at all. split_ffpp_identity_disjoint
# handles this explicitly and PRINTS the residual source-identity overlap.
official = FF.load_ffpp_official_splits(roots=("/kaggle/input",))
print("FF++ official splits:", "found" if official else "NOT found (using fallback)")

train_idx, val_idx = FF.split_ffpp_identity_disjoint(
    ROWS, seed=CONFIG["SEED"], official_splits=official, verbose=True)
assert len(train_idx) > 0 and len(val_idx) > 0
assert len({ROWS[i]["label"] for i in val_idx}) == 2, "val split needs both classes"

In [ ]:
# ---- Train. LN-tuning: ~0.03-0.05% of the backbone is trainable ----
CFG = dict(FF.FFPP_CONFIG)
CFG["MULTI_GPU"] = False   # single GPU only: nn.DataParallel was ~3x slower on CLIP (see CLAUDE.md)
# ViT-L/14 at 224px stores ~120 MB of activations per image across its 24
# layers. Gradient checkpointing (on by default in CLIPVisualBackbone) trades
# ~30% compute for ~10x less activation memory, which is what makes this fit a
# 16 GB T4 at a useful batch size.
BACKBONE = "clip_vit_l14"
OUT = "/kaggle/working/ffpp_encoder_v5.pth"

res = FF.train_ffpp_encoder(
    ROWS, train_idx, val_idx, CROP_DIR, OUT,
    backbone=BACKBONE, cfg=CFG, seed=CONFIG["SEED"], device=device, verbose=True)
model = res["model"]

In [ ]:
# ---- Zero-shot evaluation. THIS is the number that matters, not FF++ val ----
from crossfuse_v5 import build_detector

results = {}
detector = build_detector(str(device))

celebdf = []
for pat, lab in (("**/Celeb-real/**/*.mp4", 0), ("**/YouTube-real/**/*.mp4", 0),
                 ("**/Celeb-synthesis/**/*.mp4", 1)):
    celebdf += [(p, lab) for p in glob.glob(f"/kaggle/input/{pat}", recursive=True)]
if celebdf:
    rng = make_rng("celebdf_eval"); rng.shuffle(celebdf)
    celebdf = celebdf[:400]
    results["celebdf"] = FF.evaluate_ffpp_videos(
        model, celebdf, "Celeb-DF v2", detector=detector, device=device)
else:
    print("[i] Celeb-DF v2 not found -- attach it to run the gate.")

dfdc = []
for meta_path in glob.glob("/kaggle/input/**/metadata.json", recursive=True):
    folder = os.path.dirname(meta_path)
    with open(meta_path) as f:
        data = json.load(f)
    for name, info in data.items():
        fp = os.path.join(folder, name)
        if os.path.exists(fp) and info.get("label", "").upper() in ("REAL", "FAKE"):
            dfdc.append((fp, 0 if info["label"].upper() == "REAL" else 1))
if dfdc:
    rng = make_rng("dfdc_eval"); rng.shuffle(dfdc)
    dfdc = dfdc[:400]
    results["dfdc"] = FF.evaluate_ffpp_videos(
        model, dfdc, "DFDC sample", detector=detector, device=device)
else:
    print("[i] DFDC sample not found -- attach it to run the gate.")

In [ ]:
# ---- The preregistered gate ----
V5_BASELINE = {"celebdf": 0.6399, "dfdc": 0.5799}   # crossdataset_results_v5.json

print("=" * 66)
print(f"{'dataset':<16}{'v5 baseline':>13}{'this encoder':>15}{'delta':>10}{'gate':>12}")
print("-" * 66)
passed = True
for k, gate in (("celebdf", CFG["GATE_CELEBDF_AUC"]), ("dfdc", CFG["GATE_DFDC_AUC"])):
    if k not in results or results[k] is None:
        print(f"{k:<16}{'--':>13}{'not evaluated':>15}{'--':>10}{'SKIP':>12}")
        passed = False
        continue
    auc = results[k]["auc"]
    ok = auc >= gate
    passed &= ok
    print(f"{k:<16}{V5_BASELINE[k]:>13.4f}{auc:>15.4f}"
          f"{auc - V5_BASELINE[k]:>+10.4f}{('PASS' if ok else 'FAIL') + f' ({gate})':>12}")
print("=" * 66)

if passed:
    print("\nGate PASSED. Save Version, publish /kaggle/working as Kaggle Dataset "
          "'ffpp-encoder-v5', attach to NB-B, and set "
          "CONFIG['PRETRAINED_ENCODER'] to the checkpoint path there.")
else:
    print("\nGate FAILED. Do NOT proceed to NB-B on this encoder.")
    print("Report the reproduction failure rather than tuning until it passes --")
    print("that is what the gate is for. Things worth checking first:")
    print("  * did all 4 FF++ manipulation families survive extraction (NB-A3)?")
    print("  * did CLIP load PRETRAINED weights (look for the '[clip] ... tag' line)?")
    print("  * is FF++ val AUC itself high? low => underfitting, high => transfer gap.")

with open("/kaggle/working/ffpp_zeroshot_v5.json", "w") as f:
    json.dump({k: {kk: vv for kk, vv in v.items() if kk not in ("y_true", "y_score")}
               for k, v in results.items() if v}, f, indent=2)

In [ ]:
# ---- Score distributions, so a "passing" AUC with a degenerate score
# ---- distribution (everything at 0.5, ordering right by luck) is visible.
fig, axes = plt.subplots(1, max(len(results), 1), figsize=(5.5 * max(len(results), 1), 3.6),
                         squeeze=False)
for ax, (k, r) in zip(axes[0], results.items()):
    if not r:
        continue
    ys = np.asarray(r["y_score"]); yt = np.asarray(r["y_true"])
    bins = np.linspace(0, 1, 26)
    ax.hist(ys[yt == 0], bins=bins, alpha=0.6, label="real", color="#4C78A8")
    ax.hist(ys[yt == 1], bins=bins, alpha=0.6, label="fake", color="#E45756")
    ax.set_title(f"{r['name']}  AUC {r['auc']:.3f}  n={r['n']}", fontsize=10)
    ax.set_xlabel("P(fake)"); ax.legend(fontsize=8)
axes[0][0].set_ylabel("videos")
plt.suptitle("Zero-shot score distributions (FF++-trained CLIP encoder)", fontsize=11)
plt.tight_layout(); plt.show()